# Segmentacja semantyczna guzów piersi (USG) - BUSI Dataset

Porównanie trzech architektur:
- **U-Net** – standard w medycznej segmentacji obrazów
- **DeepLabV3+** – atrous convolutions + ASPP
- **SAM** (Segment Anything Model) – zero-shot inference z point prompts

**Metryka:** IoU (Intersection over Union) + Dice Coefficient

In [ ]:
# Instalacja zależności
import subprocess, sys

packages = [
    "segmentation-models-pytorch",
    "albumentations",
    "tqdm",
    "pandas",
    "seaborn",
    "segment-anything",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

# ── Deterministyczność ─────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Ścieżki ────────────────────────────────────────────────────────────────
ROOT = Path("Dataset_BUSI_with_GT")
CLASSES = ["benign", "malignant", "normal"]
IMG_SIZE = 256

## 1. Wczytanie i eksploracja danych

In [ ]:
def collect_samples(root: Path) -> list[dict]:
    """
    Zbiera pary (obraz, maska) z folderu Dataset_BUSI_with_GT.
    Każdy obraz może mieć jedną maskę lub kilka (_mask, _mask_1, ...).
    Scalamy wiele masek binarnym OR.
    """
    samples = []
    mask_re = re.compile(r"_mask")

    for cls in CLASSES:
        cls_dir = root / cls
        if not cls_dir.exists():
            continue

        # Grupowanie: nazwa_bazowa -> lista masek
        images, masks_map = {}, defaultdict(list)
        for f in sorted(cls_dir.iterdir()):
            name = f.stem
            if mask_re.search(name):
                base = mask_re.split(name)[0]
                masks_map[base].append(f)
            else:
                images[name] = f

        for base, img_path in images.items():
            if base in masks_map:
                samples.append({
                    "image": img_path,
                    "masks": masks_map[base],
                    "label": cls,
                    "label_id": CLASSES.index(cls),
                })

    return samples


samples = collect_samples(ROOT)
df = pd.DataFrame(samples)
print(f"Łączna liczba próbek: {len(samples)}")
print(df["label"].value_counts())

In [ ]:
def load_sample(sample: dict, size: int = IMG_SIZE) -> tuple[np.ndarray, np.ndarray]:
    """Zwraca (H,W,3) uint8  i  (H,W) uint8 {0,1}."""
    img = np.array(Image.open(sample["image"]).convert("RGB").resize((size, size)))
    mask = np.zeros((size, size), dtype=np.uint8)
    for mp in sample["masks"]:
        m = np.array(Image.open(mp).convert("L").resize((size, size)))
        mask = np.clip(mask + (m > 127).astype(np.uint8), 0, 1)
    return img, mask


# ── Podgląd przykładowych próbek ────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for row, cls in enumerate(CLASSES):
    cls_samples = df[df["label"] == cls].sample(2, random_state=SEED)
    for col_offset, (_, s) in enumerate(cls_samples.iterrows()):
        img, mask = load_sample(s)
        axes[row, col_offset * 2].imshow(img)
        axes[row, col_offset * 2].set_title(f"{cls} – obraz")
        axes[row, col_offset * 2].axis("off")
        axes[row, col_offset * 2 + 1].imshow(mask, cmap="gray")
        axes[row, col_offset * 2 + 1].set_title(f"{cls} – maska")
        axes[row, col_offset * 2 + 1].axis("off")
plt.suptitle("Przykładowe obrazy USG i maski GT", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Dataset, augmentacja i DataLoadery

In [ ]:
from sklearn.model_selection import train_test_split

# ── Podział 70/15/15 (stratify po klasie) ───────────────────────────────────
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["label"])
val_df, test_df  = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"])

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
print("Train distribution:\n", train_df["label"].value_counts())

# ── Augmentacje ─────────────────────────────────────────────────────────────
MEAN = (0.485, 0.456, 0.406)   # ImageNet
STD  = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
    A.RandomBrightnessContrast(p=0.4),
    A.GaussNoise(p=0.3),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])


class BUSIDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.samples = dataframe.to_dict("records")
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img, mask = load_sample(s)
        if self.transform:
            out = self.transform(image=img, mask=mask)
            img  = out["image"].float()      # (3, H, W)
            mask = out["mask"].long()        # (H, W)
        return img, mask


BATCH = 8
NUM_WORKERS = 0

train_ds = BUSIDataset(train_df, train_transform)
val_ds   = BUSIDataset(val_df,   val_transform)
test_ds  = BUSIDataset(test_df,  val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("DataLoadery gotowe.")

## 3. Metryki i funkcja pomocnicza treningu

In [ ]:
def iou_score(pred_mask: torch.Tensor, true_mask: torch.Tensor, eps: float = 1e-6) -> float:
    """
    pred_mask: (N, H, W) bool/int  OR  (H, W)
    true_mask: (N, H, W) bool/int  OR  (H, W)
    Zwraca średnie IoU po batchu.
    """
    pred = pred_mask.bool().flatten(1)
    true = true_mask.bool().flatten(1)
    inter = (pred & true).float().sum(1)
    union = (pred | true).float().sum(1)
    iou = (inter + eps) / (union + eps)
    return iou.mean().item()


def dice_score(pred_mask: torch.Tensor, true_mask: torch.Tensor, eps: float = 1e-6) -> float:
    pred = pred_mask.bool().flatten(1).float()
    true = true_mask.bool().flatten(1).float()
    inter = (pred * true).sum(1)
    denom = pred.sum(1) + true.sum(1)
    dice = (2 * inter + eps) / (denom + eps)
    return dice.mean().item()


# ── Funkcja treningu jednej epoki ───────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_iou, total_dice = 0, 0, 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)                       # (N, 1, H, W)
        preds = preds.squeeze(1)                  # (N, H, W)
        loss = criterion(preds, masks.float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            bin_preds = (preds.sigmoid() > 0.5).long()
            total_loss += loss.item()
            total_iou  += iou_score(bin_preds, masks)
            total_dice += dice_score(bin_preds, masks)
    n = len(loader)
    return total_loss / n, total_iou / n, total_dice / n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_iou, total_dice = 0, 0, 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs).squeeze(1)
        loss = criterion(preds, masks.float())
        bin_preds = (preds.sigmoid() > 0.5).long()
        total_loss += loss.item()
        total_iou  += iou_score(bin_preds, masks)
        total_dice += dice_score(bin_preds, masks)
    n = len(loader)
    return total_loss / n, total_iou / n, total_dice / n


def train_model(model, train_loader, val_loader, epochs: int = 30,
                lr: float = 1e-4, weight_decay: float = 1e-5, patience: int = 7):
    """Trening z early stopping. Zwraca słownik historii."""
    criterion = smp.losses.DiceLoss(mode="binary") + smp.losses.SoftBCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {"train_loss": [], "val_loss": [], "train_iou": [], "val_iou": [],
               "train_dice": [], "val_dice": []}
    best_iou, no_improve = 0.0, 0
    best_weights = None

    for epoch in range(1, epochs + 1):
        tr_loss, tr_iou, tr_dice = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        va_loss, va_iou, va_dice = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()

        history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
        history["train_iou"].append(tr_iou);   history["val_iou"].append(va_iou)
        history["train_dice"].append(tr_dice); history["val_dice"].append(va_dice)

        if va_iou > best_iou:
            best_iou = va_iou
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs}  "
                  f"loss={tr_loss:.4f}/{va_loss:.4f}  "
                  f"IoU={tr_iou:.4f}/{va_iou:.4f}  "
                  f"Dice={tr_dice:.4f}/{va_dice:.4f}")

        if no_improve >= patience:
            print(f"  Early stopping po {epoch} epokach (best val IoU={best_iou:.4f})")
            break

    model.load_state_dict(best_weights)
    return history


print("Funkcje gotowe.")

## 4. Model 1 – U-Net (encoder: ResNet34, pretrained ImageNet)

In [ ]:
unet_model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None,          # logity – loss sam stosuje sigmoid
).to(DEVICE)

total_params = sum(p.numel() for p in unet_model.parameters())
print(f"U-Net parametry: {total_params:,}")

print("\n--- Trening U-Net ---")
unet_history = train_model(unet_model, train_loader, val_loader, epochs=40, lr=1e-4, patience=8)

In [ ]:
def plot_history(history: dict, title: str = ""):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"],   label="val")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(history["train_iou"], label="train")
    axes[1].plot(history["val_iou"],   label="val")
    axes[1].set_title("IoU"); axes[1].legend(); axes[1].grid(True)
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()

plot_history(unet_history, "U-Net – krzywe treningu")

## 5. Model 2 – DeepLabV3+ (encoder: ResNet50, pretrained ImageNet)

In [ ]:
deeplab_model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None,
).to(DEVICE)

total_params = sum(p.numel() for p in deeplab_model.parameters())
print(f"DeepLabV3+ parametry: {total_params:,}")

print("\n--- Trening DeepLabV3+ ---")
deeplab_history = train_model(deeplab_model, train_loader, val_loader, epochs=40, lr=5e-5, patience=8)

In [ ]:
plot_history(deeplab_history, "DeepLabV3+ – krzywe treningu")

## 6. Model 3 – SAM (Segment Anything Model) – zero-shot inference

SAM jest modelem *foundation* trenowanym przez Meta na ~1 mld masek.  
Używamy go w trybie **zero-shot** z punktowym promptem:  
- dla obrazów z maską GT – punkt promptu = centroid GT maski  
- dla obrazów normalnych (brak guza) – punkt w centrum obrazu

Pobieramy checkpoint `sam_vit_b` (~375 MB, najlżejszy wariant).

In [ ]:
import urllib.request
from pathlib import Path

SAM_CHECKPOINT = Path("sam_vit_b_01ec64.pth")
SAM_URL = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

if not SAM_CHECKPOINT.exists():
    print(f"Pobieranie SAM checkpoint ({SAM_URL}) ...")
    urllib.request.urlretrieve(SAM_URL, SAM_CHECKPOINT)
    print("Pobrano!")
else:
    print(f"Checkpoint już istnieje: {SAM_CHECKPOINT}")

print(f"Rozmiar pliku: {SAM_CHECKPOINT.stat().st_size / 1e6:.1f} MB")

In [ ]:
from segment_anything import sam_model_registry, SamPredictor

sam = sam_model_registry["vit_b"](checkpoint=str(SAM_CHECKPOINT))
sam.to(DEVICE)
sam_predictor = SamPredictor(sam)
print("SAM załadowany.")


def get_centroid(mask_np: np.ndarray) -> tuple[int, int]:
    """Zwraca (x, y) centroidu maski lub centrum obrazu jeśli maska jest pusta."""
    ys, xs = np.where(mask_np > 0)
    if len(xs) == 0:
        h, w = mask_np.shape
        return w // 2, h // 2
    return int(xs.mean()), int(ys.mean())


@torch.no_grad()
def sam_inference_on_loader(predictor: SamPredictor,
                            test_df_samples: list[dict],
                            size: int = IMG_SIZE) -> tuple[float, float]:
    """
    Uruchamia SAM na próbkach z test_df.
    Punkt promptu = centroid GT maski (symulacja kliknięcia lekarza).
    Zwraca (mean_iou, mean_dice).
    """
    ious, dices = [], []
    for s in tqdm(test_df_samples, desc="SAM inference"):
        img_rgb, gt_mask = load_sample(s, size=size)
        predictor.set_image(img_rgb)

        cx, cy = get_centroid(gt_mask)
        input_point = np.array([[cx, cy]])
        input_label = np.array([1])          # punkt foreground

        masks, scores, _ = predictor.predict(
            point_coords=input_point,
            point_labels=input_label,
            multimask_output=True,
        )
        # Wybierz maskę z najwyższym score
        best_idx = np.argmax(scores)
        pred_mask = masks[best_idx].astype(np.uint8)

        # Oblicz metryki
        pred_t = torch.from_numpy(pred_mask).unsqueeze(0)
        true_t = torch.from_numpy(gt_mask).unsqueeze(0)
        ious.append(iou_score(pred_t, true_t))
        dices.append(dice_score(pred_t, true_t))

    return float(np.mean(ious)), float(np.mean(dices))


print("Funkcje SAM gotowe.")

In [ ]:
test_samples = test_df.to_dict("records")
sam_iou, sam_dice = sam_inference_on_loader(sam_predictor, test_samples)
print(f"SAM  →  IoU={sam_iou:.4f}  Dice={sam_dice:.4f}")

## 7. Ewaluacja na zbiorze testowym

In [ ]:
dummy_criterion = smp.losses.DiceLoss(mode="binary") + smp.losses.SoftBCEWithLogitsLoss()

_, unet_iou_test,    unet_dice_test    = evaluate(unet_model,    test_loader, dummy_criterion, DEVICE)
_, deeplab_iou_test, deeplab_dice_test = evaluate(deeplab_model, test_loader, dummy_criterion, DEVICE)

print(f"U-Net      →  IoU={unet_iou_test:.4f}  Dice={unet_dice_test:.4f}")
print(f"DeepLabV3+ →  IoU={deeplab_iou_test:.4f}  Dice={deeplab_dice_test:.4f}")
print(f"SAM (zero-shot) →  IoU={sam_iou:.4f}  Dice={sam_dice:.4f}")

In [ ]:
# ── Tabela wyników ─────────────────────────────────────────────────────────
results_df = pd.DataFrame({
    "Model":         ["U-Net (ResNet34)", "DeepLabV3+ (ResNet50)", "SAM vit_b (zero-shot)"],
    "IoU (test)":    [round(unet_iou_test, 4), round(deeplab_iou_test, 4), round(sam_iou, 4)],
    "Dice (test)":   [round(unet_dice_test, 4), round(deeplab_dice_test, 4), round(sam_dice, 4)],
    "Typ":           ["Supervised (fine-tune)", "Supervised (fine-tune)", "Zero-shot"],
})
print(results_df.to_string(index=False))

# ── Wykres słupkowy ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
palette = ["#4C72B0", "#DD8452", "#55A868"]

axes[0].bar(results_df["Model"], results_df["IoU (test)"],  color=palette)
axes[0].set_title("IoU na zbiorze testowym"); axes[0].set_ylim(0, 1)
axes[0].set_xticklabels(results_df["Model"], rotation=12, ha="right")
axes[0].grid(axis="y", alpha=0.4)

axes[1].bar(results_df["Model"], results_df["Dice (test)"], color=palette)
axes[1].set_title("Dice na zbiorze testowym"); axes[1].set_ylim(0, 1)
axes[1].set_xticklabels(results_df["Model"], rotation=12, ha="right")
axes[1].grid(axis="y", alpha=0.4)

plt.tight_layout(); plt.show()

## 8. Wizualizacja predykcji na przykładowych obrazach

In [ ]:
@torch.no_grad()
def predict_smp(model, img_rgb: np.ndarray, device) -> np.ndarray:
    """Zwraca binarną maskę (H, W) uint8."""
    aug = val_transform(image=img_rgb, mask=np.zeros(img_rgb.shape[:2], dtype=np.uint8))
    img_t = aug["image"].unsqueeze(0).float().to(device)
    logit = model(img_t).squeeze().cpu()
    return (logit.sigmoid() > 0.5).numpy().astype(np.uint8)


def predict_sam(predictor: SamPredictor, img_rgb: np.ndarray, gt_mask: np.ndarray) -> np.ndarray:
    predictor.set_image(img_rgb)
    cx, cy = get_centroid(gt_mask)
    masks, scores, _ = predictor.predict(
        point_coords=np.array([[cx, cy]]),
        point_labels=np.array([1]),
        multimask_output=True,
    )
    return masks[np.argmax(scores)].astype(np.uint8)


# ── Losowe 4 próbki z test setu ─────────────────────────────────────────────
sample_indices = random.sample(range(len(test_samples)), min(4, len(test_samples)))

fig, axes = plt.subplots(len(sample_indices), 5, figsize=(18, 4 * len(sample_indices)))
if len(sample_indices) == 1:
    axes = axes[np.newaxis, :]

col_titles = ["Obraz oryginalny", "GT Maska", "U-Net", "DeepLabV3+", "SAM (zero-shot)"]
for ax, ct in zip(axes[0], col_titles):
    ax.set_title(ct, fontsize=11)

for row, idx in enumerate(sample_indices):
    s = test_samples[idx]
    img_rgb, gt_mask = load_sample(s)

    unet_pred    = predict_smp(unet_model,    img_rgb, DEVICE)
    deeplab_pred = predict_smp(deeplab_model, img_rgb, DEVICE)
    sam_pred     = predict_sam(sam_predictor, img_rgb, gt_mask)

    # Resize predykcji do IMG_SIZE jeśli potrzeba
    unet_pred    = cv2.resize(unet_pred,    (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    deeplab_pred = cv2.resize(deeplab_pred, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    sam_pred     = cv2.resize(sam_pred,     (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

    iou_u  = iou_score(torch.from_numpy(unet_pred).unsqueeze(0),    torch.from_numpy(gt_mask).unsqueeze(0))
    iou_d  = iou_score(torch.from_numpy(deeplab_pred).unsqueeze(0), torch.from_numpy(gt_mask).unsqueeze(0))
    iou_s  = iou_score(torch.from_numpy(sam_pred).unsqueeze(0),     torch.from_numpy(gt_mask).unsqueeze(0))

    axes[row, 0].imshow(img_rgb);    axes[row, 0].axis("off")
    axes[row, 1].imshow(gt_mask, cmap="gray"); axes[row, 1].axis("off")
    axes[row, 2].imshow(unet_pred,    cmap="gray"); axes[row, 2].axis("off"); axes[row, 2].set_xlabel(f"IoU={iou_u:.3f}")
    axes[row, 3].imshow(deeplab_pred, cmap="gray"); axes[row, 3].axis("off"); axes[row, 3].set_xlabel(f"IoU={iou_d:.3f}")
    axes[row, 4].imshow(sam_pred,     cmap="gray"); axes[row, 4].axis("off"); axes[row, 4].set_xlabel(f"IoU={iou_s:.3f}")

    label = s["label"]
    axes[row, 0].set_ylabel(f"{label}", fontsize=10, rotation=0, labelpad=55)

plt.suptitle("Porównanie predykcji: U-Net vs DeepLabV3+ vs SAM", fontsize=13)
plt.tight_layout(); plt.show()

## 9. Analiza per-klasa (benign / malignant / normal)

In [ ]:
rows = []
for s in tqdm(test_samples, desc="Per-class eval"):
    img_rgb, gt_mask = load_sample(s)
    cls = s["label"]

    unet_pred    = predict_smp(unet_model,    img_rgb, DEVICE)
    deeplab_pred = predict_smp(deeplab_model, img_rgb, DEVICE)
    sam_pred     = predict_sam(sam_predictor, img_rgb, gt_mask)

    unet_pred    = cv2.resize(unet_pred,    (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    deeplab_pred = cv2.resize(deeplab_pred, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    sam_pred     = cv2.resize(sam_pred,     (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

    gt_t = torch.from_numpy(gt_mask).unsqueeze(0)
    rows.append({
        "label":        cls,
        "IoU_UNet":     iou_score(torch.from_numpy(unet_pred).unsqueeze(0),    gt_t),
        "IoU_DeepLab":  iou_score(torch.from_numpy(deeplab_pred).unsqueeze(0), gt_t),
        "IoU_SAM":      iou_score(torch.from_numpy(sam_pred).unsqueeze(0),     gt_t),
        "Dice_UNet":    dice_score(torch.from_numpy(unet_pred).unsqueeze(0),   gt_t),
        "Dice_DeepLab": dice_score(torch.from_numpy(deeplab_pred).unsqueeze(0),gt_t),
        "Dice_SAM":     dice_score(torch.from_numpy(sam_pred).unsqueeze(0),    gt_t),
    })

per_class_df = pd.DataFrame(rows)

print("\n=== IoU per klasa ===")
print(per_class_df.groupby("label")[["IoU_UNet","IoU_DeepLab","IoU_SAM"]].mean().round(4))
print("\n=== Dice per klasa ===")
print(per_class_df.groupby("label")[["Dice_UNet","Dice_DeepLab","Dice_SAM"]].mean().round(4))

In [ ]:
# ── Heatmapa IoU per klasa ─────────────────────────────────────────────────
iou_pivot = per_class_df.groupby("label")[["IoU_UNet","IoU_DeepLab","IoU_SAM"]].mean()
iou_pivot.columns = ["U-Net", "DeepLabV3+", "SAM"]

fig, ax = plt.subplots(figsize=(7, 3))
sns.heatmap(iou_pivot, annot=True, fmt=".3f", cmap="YlOrRd",
            vmin=0, vmax=1, ax=ax, linewidths=0.5)
ax.set_title("IoU per klasa – porównanie modeli")
plt.tight_layout(); plt.show()

# Boxplot
melted = per_class_df.melt(id_vars="label",
                            value_vars=["IoU_UNet","IoU_DeepLab","IoU_SAM"],
                            var_name="Model", value_name="IoU")
melted["Model"] = melted["Model"].map({"IoU_UNet":"U-Net","IoU_DeepLab":"DeepLabV3+","IoU_SAM":"SAM"})

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=melted, x="label", y="IoU", hue="Model", ax=ax, palette="Set2")
ax.set_title("Rozkład IoU per klasa"); ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

## 10. Wnioski końcowe

In [ ]:
print("="*60)
print("PODSUMOWANIE – Segmentacja guzów piersi (BUSI)")
print("="*60)
print(results_df.to_string(index=False))
print()

best_model = results_df.loc[results_df["IoU (test)"].idxmax(), "Model"]
print(f"Najlepszy model wg IoU: {best_model}")
print()
print("Obserwacje:")
print("  • U-Net i DeepLabV3+ są modelami nadzorowanymi fine-tune'owanymi na BUSI.")
print("  • SAM działa zero-shot – nie widział żadnych obrazów USG podczas treningu.")
print("  • Transfer learning z ImageNet jest kluczowy dla małego zbioru (~500 próbek).")
print("  • Klasa 'normal' ma puste maski – modele uczą się przewidywać tło (IoU ≈ 1).")
print("  • SAM sprawdza się jako silny baseline; może być dalej fine-tune'owany (SAM-Med2D).")